# Convert the CSV Data to RDF

This notebook is used to convert the CSV data into RDF format. The CSV files are read and processed, and the resulting RDF data is saved in Turtle format. The code is intentionally very verbose and not optimized for performance, as the main goal is to have a clear and understandable code that can be easily modified and extended.

## Create Dictionary

In the following part, all information from the different CSV files is combined into a **single nested dictionary** structure. This is done by iterating over the rows of the `party_csv` (which is the "main" file) and fetching all the related information from the other connected CSV files.

### Read CSV Files

In [1]:
from pathlib import Path
import pandas as pd

# path to all the CSV files relative to the location of this notebook
CSV_PATH = Path("../data/input")

# main file
party_csv = pd.read_csv(CSV_PATH / "party.csv")

# helper files
party_name_csv = pd.read_csv(CSV_PATH / "party_name.csv")
relationship_csv = pd.read_csv(CSV_PATH / "relationship.csv")
relationship_type_csv = pd.read_csv(CSV_PATH / "relationship_type.csv")
party_code_csv = pd.read_csv(CSV_PATH / "party_code.csv")
party_code_type_csv = pd.read_csv(CSV_PATH / "party_code_type.csv")

In [2]:
display(party_csv)

,party_id,identifier,color,party_name_id,relationship_id_1,relationship_id_2,social_group_id,party_category_id,election_result_id,period_of_service_id,...,founding_date_datetime,founding_date_notes,founding_date_sources,dissolution_date_precision,dissolution_date_datetime,dissolution_date_notes,dissolution_date_sources,translations_languages_code3,translations_description3,translations_hls_link_external_url3
0,1,svp-ch,#4B8A3E,[1],"[3,9]","[5,12,35,69,90,98,141,174,192,193,232,260,276,...",[1],[16],"[1,2,3,4,5,6,7,8,9,10,11,12,13,14,106,107,108,...","[1,2,3,4,5,6,7,8,9,562,563,564,565,566,567,568...",...,1971-09-22T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,sp-ch,#F0554D,[2],"[618,635]","[13,36,70,91,99,142,175,196,201,202,233,234,26...",[2],[1],"[15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,...","[10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,...",...,1888-10-21T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,fdplib-ch,#3B79BF,[3],"[1,6]","[18,38,72,93,101,135,144,177,203,205,210,240,2...",[3],[15],"[43,44,45,46,148,149,150,151,6290,7226,7227,72...","[38,39,40,41]",...,2009-01-01T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,mitte-ch,#D6862B,[4],"[2,4]","[21,31,40,43,61,75,95,103,125,145,146,163,180,...",[4],[14],"[47,152,6284,7250,7251,7252,7253,10247]","[42,43,611,612]",...,2021-01-01T12:00:00,NaN,page_number_string: 360\r\nnotes: \r\nsource_f...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,gruene-ch,#84B547,"[6,7,8,9,199]",[],"[22,44,45,48,49,53,54,56,78,79,96,104,105,106,...",[5],[13],"[48,49,50,51,52,53,54,55,56,57,58,59,60,153,15...","[44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59]",...,1983-05-28T12:00:00,NaN,page_number_string: 3\r\nnotes: \r\nsource_ful...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,421,freiw-ch,NaN,[671],[],[],[417],[416],[],[],...,1943-09-15T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,year,1990-07-01T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,NaN,NaN,NaN
416,422,dsp-ch,NaN,[672],[638],"[639,640,641,644]",[418],[417],[],[],...,1994-06-04T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,unknown,2012-01-01T12:00:00,note_translations: languages_code: de-CH\r\nno...,NaN,NaN,NaN,NaN
417,423,dsp-bl,NaN,[673],[640],[],[419],[418],[],[],...,1995-09-23T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,unknown,2010-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN
418,424,dsp-gr,NaN,[674],"[641,642]",[],[420],[419],"[14802,14803,14804,14805,14806,14807,14808]",[],...,1987-11-03T12:00:00,NaN,page_number_string: \r\nnotes: \r\nsource_full...,unknown,2017-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN


### Date Handling

All the dates in the CSV and after import to a pandas DataFrame are given as strings in the form of "YYYY-MM-DDT12:00:00" format (always with the time set to 12:00:00). The time part is not relevant for our use case, so we will remove it. In the final RDF data, the end dates will be shifted by one day as we are using the `schema:validThrough` property, which means the last valid day.

- `schema:validFrom` means the first valid day
- `schema:validThrough` means the last valid day

Unknown dates are represented as `NaN` in the DataFrame. To check for unknown dates, we will use the `pd.isna()` function instead of checking for `None`, as the latter does not work with `NaN` values.

In [3]:
def convert_time_column(df, column_name):
    """
    Convert the time column to a date format and remove the time part. 
    Convert it back to strings in the format YYYY-MM-DD.
    """
    df[column_name] = pd.to_datetime(df[column_name], errors='coerce').dt.date
    df[column_name] = df[column_name].astype(str)
    return df

In [4]:
# convert the time columns to date format and remove the time part

party_csv = convert_time_column(party_csv, 'founding_date_datetime')
party_csv = convert_time_column(party_csv, 'dissolution_date_datetime')
party_name_csv = convert_time_column(party_name_csv, 'from_date_datetime')
party_name_csv = convert_time_column(party_name_csv, 'to_date_datetime')
party_relationship_csv = convert_time_column(relationship_csv, 'from_date_datetime')
party_relationship_csv = convert_time_column(relationship_csv, 'to_date_datetime')

### Create Parties Dictionary

In [5]:
def parse_list(s):
    """
    Parse a string representation of a list like "[1, 2, 3]" into a Python list.
    """
    if pd.isna(s) or s == "[]":
        return []
    return [int(item) for item in s.strip("[]").split(",")]

In [6]:
def is_currently_valid(day, valid_from, valid_through):
    """
    Check if a given day is within the valid range defined by valid_from and valid_through.
    valid_through can be NaN to indicate that the validity is open-ended. 
    """
    return valid_from <= str(day) and (pd.isna(valid_through) or str(day) < valid_through)

In [27]:
# create the main dictionary that will hold all the parties and their information
parties = {}

# for each identity (each row in the party_csv)
for _, row in party_csv.iterrows():
    
    # Create a dictionary for the current party
    party = {}
    
    # get the party_id - needed to set the key in the parties dictionary
    party_id = row["party_id"]

    # basic date information
    party["founding_date"] = row["founding_date_datetime"]
    party["dissolution_date"] = row["dissolution_date_datetime"]

    
    #### names ####
    
    # get party names
    party_name_ids = parse_list(row["party_name_id"])

    # all names are stored in a list of dictionaries
    names = []

    for party_name_id in party_name_ids:
        
        # names
        name_lang = {}
        for lang_id in range(1, 5):  # there are 4 language columns (1, 2, 3, 4)
            lang_name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_languages_code{lang_id}"].item()
            name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_name{lang_id}"].item()
            if str(lang_name) != "nan" and str(name) != "nan":
                name_lang[str(lang_name)[0:2]] = name
        
        # abbreviations
        abbr_lang = {}
        for lang_id in range(1, 5):  # there are 4 language columns (1, 2, 3, 4)
            lang_abbr = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_languages_code{lang_id}"].item()
            abbr = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_abbreviation{lang_id}"].item()
            if str(lang_abbr) != "nan" and str(abbr) != "nan":
                abbr_lang[str(lang_abbr)[0:2]] = abbr
        
        # dates for the name and abbreviation
        valid_from = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "from_date_datetime"].item()
        valid_through = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "to_date_datetime"].item()
        
        # create a name entry and add it to the names list
        name_entry = {"id": party_name_id, "name": name_lang, "abbr": abbr_lang, "valid_from": valid_from, "valid_through": valid_through}
        names.append(name_entry)

    
    # sort names by from_date
    names.sort(key=lambda x: x["valid_from"])
    
    # add names to the party dictionary
    party["names"] = names

    
    #### relationships ####

    # get relationships, they are given in two columns, relationship_id_1 and relationship_id_2.
    # They go to either shape-changing or hierarchical relationships.
    relationships = []

    shape_changing_relationship_types = ["successor-of", "split-from", "accessed"]
    hierarchical_relationship_types = ["child-of", "observer-in", "affiliated-with"]

    # get relationships from both columns and combine them into a single list
    relationship_ids = parse_list(row["relationship_id_1"]) + parse_list(row["relationship_id_2"])

    for relationship_id in relationship_ids:
        
        relationship_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_1_id"].item()
        relationship_to = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_2_id"].item()
        relationship_type = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "type"].item()
        relationship_label = relationship_type_csv.loc[relationship_type_csv["relationship_type_id"] == relationship_type, "identifier"].item()
        valid_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "from_date_datetime"].item()
        valid_through = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "to_date_datetime"].item()
        
        relationship_entry = {"id": relationship_id, "type_label": relationship_label,"party_from": relationship_from, "party_to": relationship_to, "valid_from": valid_from, "valid_through": valid_through}
        
        if relationship_entry["type_label"] in shape_changing_relationship_types:
            relationship_entry["supra_type_label"] = "shape-changing"
            relationships.append(relationship_entry)
        else:
            # hierarchical relationships are only added on one side because of "version-avalanche" effect.
            if relationship_entry["party_to"] != party_id:
                relationship_entry["supra_type_label"] = "hierarchical"
                relationships.append(relationship_entry)

    # sort relationships by from_date and add them to the party dictionary
    relationships.sort(key=lambda x: x["valid_from"])
    
    # add relationships to the party dictionary
    party["relationships"] = relationships

    
    #### party identifiers ####

    # different identifiers for the same party (e.g. bfs-code, wikidata, etc.)
    party_code_ids = parse_list(row["party_code_id"])
    party_codes = []

    for party_code_id in party_code_ids:
        code = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "code"].item()
        code_type = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "party_code_type_id"].item()
        code_type_label = party_code_type_csv.loc[party_code_type_csv["party_code_type_id"] == code_type, "identifier"].item()

        code_entry = {"id": party_code_id, "code": code, "type_label": code_type_label}
        party_codes.append(code_entry)

    party_codes.append({"code": row["identifier"], "type_label": "ap-identifier"})  # add the identifier from the party.csv row

    party["party_codes"] = party_codes


    #### party color ####
    
    party["color"] = row["color"]  # add the color from the party.csv row


    #### events ####

    events = []

    # add name change events
    for name in names:

        # name start event
        events.append({
            "event_type": "name-change_start",
            "corresponding_id": name["id"],
            "date": name["valid_from"],
        })
        # name stop event if valid_through is not NaN
        if not pd.isna(name["valid_through"]):
            events.append({
                "event_type": "name-change_stop",
                "corresponding_id": name["id"],
                "date": name["valid_through"],
            })

    # add relationship events
    for relationship in relationships:

        # if the relationship is shape-changing, this is an instantaneous event, so we only add one event for it.
        if relationship["supra_type_label"] == "shape-changing":
            events.append({
                "event_type": relationship["type_label"],
                "event_supra_type": relationship["supra_type_label"],
                "corresponding_id": relationship["id"],
                "date": relationship["valid_from"],
                "party_from": relationship["party_from"],
                "party_to": relationship["party_to"]
            })

        # hierarchical relationships are ongoing events, so we add a start and stop event (if applicable) for them.
        else:
            events.append({
                "event_type": f"{relationship["type_label"]}_start",
                "event_supra_type": relationship["supra_type_label"],
                "corresponding_id": relationship["id"],
                "date": relationship["valid_from"],
                "party_from": relationship["party_from"],
                "party_to": relationship["party_to"]
            })
            if not pd.isna(relationship["valid_through"]):
                events.append({
                    "event_type": f"{relationship["type_label"]}_stop",
                    "event_supra_type": relationship["supra_type_label"],
                    "corresponding_id": relationship["id"],
                    "date": relationship["valid_through"],
                    "party_from": relationship["party_from"],
                    "party_to": relationship["party_to"]
                })

    # sort events by date
    events.sort(key=lambda x: x["date"])

    # add founding event at position 0 (first event)
    events.insert(0, {
        "event_type": "founding",
        "date": party["founding_date"],
    })

    # add dissolution event if it exists at last position (last event)
    if not pd.isna(party["dissolution_date"]):
        events.append({
            "event_type": "dissolution",
            "date": party["dissolution_date"],
        })

    # events will only be added to the party dictionary after the mapping 
    # to started and stopped versions is done, because the version_ids are 
    # needed for that mapping. 


    #### event_dates and version_ids #### 

    # list all unique event dates in sorted order, this will be used to create version_ids
    event_dates = sorted(set([event["date"] for event in events]))
    party["event_dates"] = event_dates

    version_ids = event_dates.copy()

    # if the last event is a dissolution event, there is no version after that
    if not pd.isna(party["dissolution_date"]):
        version_ids.remove(party["dissolution_date"])

    party["version_ids"] = version_ids


    #### map events to started or stopped versions ####

    for event in events:

        # map the event to the version that started or stopped with this event.
        
        event_index = event_dates.index(event["date"])
        if event_index == 0:
            version_before_event = None  # there is no version before the first event
        else:
            version_before_event = version_ids[event_index - 1]  # the version that was valid before this event

        try:
            version_after_event = version_ids[event_dates.index(event["date"])]  # the version that will be valid after this event
        except IndexError:
            version_after_event = None

        event["started_version"] = version_after_event
        event["stopped_version"] = version_before_event


        if event["event_type"] == "successor-of" and event["party_from"] != party_id:
            event["outside_successor"] = f"{event["party_from"]}_{event["date"]}"  # add the other successor party id to the event

        if event["event_type"] == "split-from" and event["party_from"] != party_id:
            event["outside_successor"] = f"{event["party_from"]}_{event["date"]}"  # add the other successor party id to the event

        if event["event_type"] == "accessed":
            event["outside_successor"] = f"{event["party_from"]}_{event["date"]}"  # add the other successor party id to the event
            
    # add events to the party dictionary
    party["events"] = events


    #### versions ####

    versions = {}

    for i, version_id in enumerate(version_ids):
        versions[version_id] = {
            "valid_from": version_id,
            "valid_through": None,
            "is_founding": False,
            "is_dissolution": False,
            "name": None,
            "abbr": None,
            "relationships": [],
            "successors": []
        }

        # if there is a next version, set the valid_through to the next version's date
        if i < len(version_ids) - 1:
            versions[version_id]["valid_through"] = version_ids[i + 1]

            # regular successor within the same party
            versions[version_id]["successors"].append(f"{party_id}_{version_ids[i + 1]}")

        # if it is the last version, one has to check if there is another event 
        # which would terminate the version, e.g. a dissolution event or a name-change_stop event. 
        # If there is no such event, the valid_through is None, meaning the version is still 
        # ongoing.
        else:
            if len(event_dates) > len(version_ids):
                versions[version_id]["valid_through"] = event_dates[-1]

        # check for outside successors (to other parties) in the events
        for event in events:
            if event["stopped_version"] == version_id and "outside_successor" in event:
                versions[version_id]["successors"].append(event["outside_successor"])

        # add the currently valid name
        for name in names:
            if is_currently_valid(version_id, name["valid_from"], name["valid_through"]):
                versions[version_id]["name"] = name["name"]
                versions[version_id]["abbr"] = name["abbr"]
                break  # only one name should be valid at a time

        # add the currently valid hierarchical relationships
        versions[version_id]["relationships"] = []
        hierarchical_relationships = [r for r in relationships if r["supra_type_label"] == "hierarchical"]

        for relationship in hierarchical_relationships:
            if is_currently_valid(version_id, relationship["valid_from"], relationship["valid_through"]):
                versions[version_id]["relationships"].append({
                    "type_label": relationship["type_label"],
                    "party_to": relationship["party_to"]
                })

        # add the starting and stopping events for this version
        versions[version_id]["starting_events"] = [event for event in events if event["started_version"] == version_id]

        # add a flag to indicate if this version is the founding version (if there is a founding event in the starting events)
        if [event for event in versions[version_id]["starting_events"] if event["event_type"] == "founding"]:
            versions[version_id]["is_founding"] = True

        versions[version_id]["stopping_events"] = [event for event in events if event["stopped_version"] == version_id]

        # add a flag to indicate if this version is the dissolution version (if there is a dissolution event in the stopping events)
        if [event for event in versions[version_id]["stopping_events"] if event["event_type"] == "dissolution"]:
            versions[version_id]["is_dissolution"] = True

    # add party versions to the party dictionary
    party["versions"] = versions

    # add party to parties dictionary (finish line operations)
    parties[party_id] = party


### Check Dictionary Structure

In [28]:
import json

print(json.dumps(parties, indent=2, ensure_ascii=False))

{
  "1": {
    "founding_date": "1971-09-22",
    "dissolution_date": NaN,
    "names": [
      {
        "id": 1,
        "name": {
          "de": "Schweizerische Volkspartei",
          "fr": "Union Démocratique du Centre",
          "en": "Schweizerische Volkspartei"
        },
        "abbr": {
          "de": "SVP",
          "fr": "UDC"
        },
        "valid_from": "1971-09-22",
        "valid_through": NaN
      }
    ],
    "relationships": [
      {
        "id": 3,
        "type_label": "successor-of",
        "party_from": 1,
        "party_to": 11,
        "valid_from": "1971-09-22",
        "valid_through": NaN,
        "supra_type_label": "shape-changing"
      },
      {
        "id": 9,
        "type_label": "successor-of",
        "party_from": 1,
        "party_to": 18,
        "valid_from": "1971-09-22",
        "valid_through": NaN,
        "supra_type_label": "shape-changing"
      },
      {
        "id": 5,
        "type_label": "split-from",
        "party_

## Convert the Dictionary to RDF

### URIs

In [19]:
PARTY_IDENTITY = "https://politics.ld.admin.ch/party-id"
PARTY_VERSION = "https://politics.ld.admin.ch/party-version"
PARTY_VOCAB = "https://politics.schema.ld.admin.ch/party-vocab/"
PARTY_STARTING = "https://politics.schema.ld.admin.ch/starting-class"
PARTY_STOPPING = "https://politics.schema.ld.admin.ch/stopping-class"
PARTY_PREFIX = "pol"

### Transformation

In [29]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace

# create graph without binding namespaces to avoid automatic prefixing
g = Graph(bind_namespaces="none")

schema = Namespace("http://schema.org/")
g.bind("schema", schema)
xsd = Namespace("http://www.w3.org/2001/XMLSchema#")
g.bind("xsd", xsd)
owl = Namespace("http://www.w3.org/2002/07/owl#")
g.bind("owl", owl)
vl = Namespace("https://version.link/")
g.bind("vl", vl)
pol = Namespace(PARTY_VOCAB)
g.bind(PARTY_PREFIX, pol)

for party_id, party in parties.items():
    
    # create identity
    identity_uri = URIRef(f"{PARTY_IDENTITY}/{party_id}")
    g.add((identity_uri, RDF.type, vl.Identity))
    if not pd.isna(party["dissolution_date"]):
        g.add((identity_uri, RDF.type, vl.Deprecated))

    # link to the last version
    g.add((identity_uri, vl.version, URIRef(f"{PARTY_VERSION}/{party_id}_{party['version_ids'][-1]}")))

    # add wikidata identifier if it exists
    wikidata_code = next((code["code"] for code in party["party_codes"] if code["type_label"] == "wikidata-code"), None)
    if wikidata_code:
        g.add((identity_uri, owl.sameAs, URIRef(f"https://www.wikidata.org/wiki/{wikidata_code}")))

    # add color
    if not pd.isna(party["color"]):
        g.add((identity_uri, schema.color, Literal(party["color"])))

    # add founding date to the identity
    g.add((identity_uri, schema.validFrom, Literal(party["founding_date"], datatype=xsd.date)))

    # create versions
    for version_id, version in party["versions"].items():
                    
        version_uri = URIRef(f"{PARTY_VERSION}/{party_id}_{version_id}")
        g.add((version_uri, RDF.type, vl.Version))
        g.add((version_uri, vl.identity, identity_uri))

        # add dates
        g.add((version_uri, schema.validFrom, Literal(version["valid_from"], datatype=xsd.date)))
        if not pd.isna(version["valid_through"]):
            # set validThrough one day before
            valid_through = pd.to_datetime(version["valid_through"]) - pd.Timedelta(days=1)
            valid_through = str(valid_through.date())  # convert to date object
            g.add((version_uri, schema.validThrough, Literal(valid_through, datatype=xsd.date)))
            # if last version
            if version_id == party['version_ids'][-1]:
                g.add((identity_uri, schema.validThrough, Literal(valid_through, datatype=xsd.date)))

        # add name
        name_entry = version["name"]
        if not pd.isna(name_entry):
            for lang, name in name_entry.items():
                g.add((version_uri, schema.name, Literal(name, lang=lang)))
                if version_id == party['version_ids'][-1]:  # if this is the last version, also add the name to the identity
                    g.add((identity_uri, schema.name, Literal(name, lang=lang)))

        # add abbreviation
        abbr_entry = version["abbr"]
        if not pd.isna(abbr_entry):
            for lang, abbr in abbr_entry.items():
                g.add((version_uri, schema.alternateName, Literal(abbr, lang=lang)))
                if version_id == party['version_ids'][-1]:  # if this is the last version, also add the abbreviation to the identity
                    g.add((identity_uri, schema.alternateName, Literal(abbr, lang=lang)))

        # add relationships
        for relationship in version["relationships"]:
            # convert relationship type from hyphenated to camelCase for RDF property
            first, second = relationship["type_label"].split("-")
            relationship_type = first + second.capitalize()  # e.g. child-of becomes childOf
            g.add((version_uri, pol[f"{relationship_type}"], URIRef(f"{PARTY_IDENTITY}/{relationship['party_to']}")))

        # add starting classes
        for starting_event in version["starting_events"]:
            g.add((version_uri, pol.hasVersionStartingClass, URIRef(f"{PARTY_STARTING}/{starting_event["event_type"]}")))

        # add stopping classes
        for stopping_event in version["stopping_events"]:
            g.add((version_uri, pol.hasVersionStoppingClass, URIRef(f"{PARTY_STOPPING}/{stopping_event["event_type"]}")))

        # link to successors and predecessors
        for successor in version["successors"]:
            g.add((version_uri, vl.successor, URIRef(f"{PARTY_VERSION}/{successor}")))
            g.add((URIRef(f"{PARTY_VERSION}/{successor}"), vl.predecessor, version_uri))

g.serialize(destination="../data/output/parties.ttl", format="turtle")

<Graph identifier=Na6dbe9f5090447448716d22a7eba0805 (<class 'rdflib.graph.Graph'>)>